# Rerun the SEP analysis

Downloads the median-nerve recording for 6 subjects from OpenNeuro ds004388 and runs `src/sep_fixed_channels.py`. In Colab choose **Runtime > Run all**. The last cell downloads the results.

In [ ]:
!git clone -q https://github.com/harshitha-purama/sep-pipeline.git
%cd sep-pipeline
!pip install -q mne pandas scipy matplotlib pytest awscli
!python -m pytest tests -q

In [ ]:
import subprocess
subjects = [f'sub-{i:03d}' for i in range(1, 7)]
for sub in subjects:
    prefix = f's3://openneuro.org/ds004388/{sub}/eeg/'
    listing = subprocess.run(['aws', '--no-sign-request', 's3', 'ls', prefix], capture_output=True, text=True).stdout
    names = sorted(line.split()[-1] for line in listing.splitlines() if 'task-median_run-' in line and line.endswith('_eeg.set'))
    if not names:
        print(sub, 'no median run found')
        continue
    stem = names[0][:-4]
    for ext in ('.set', '.fdt'):
        subprocess.run(['aws', '--no-sign-request', 's3', 'cp', prefix + stem + ext, f'data/ds004388/{sub}/eeg/'], check=True)
    print(sub, stem)

In [ ]:
!python src/sep_fixed_channels.py --subjects sub-001 sub-002 sub-003 sub-004 sub-005 sub-006

In [ ]:
!zip -q sep_rerun.zip results/sep_fixed_channels.csv results/sep_fixed_channels_summary.csv results/figures/sep_fixed_channels.png
from google.colab import files
files.download('sep_rerun.zip')